# API da Gmail

## O que é uma API

**API** é a sigla para Application Programming Interface (Interface de Programação de Aplicações). Em termos simples, é um conjunto de regras e ferramentas que permite que diferentes softwares se comuniquem entre si.

A API:

- Recebe seu pedido

- Verifica se você tem permissão (autenticação)

- Busca os dados no servidor do Gmail

- Retorna a resposta em um formato padronizado (geralmente JSON)

Com o uso de APIs todas as interações seguem regras claras, o banco de de dados nunca é acessado direto do servidor e isso garante segurança dos usuários, você recebe somente as informações que pediu e não precisa saber como o Gmail é implementado por dentro.

Os principais componente de uma API são:

- Endpoint: é uma URL especifica para cada recurso, por exemplo, "https://gmail.googleapis.com/gmail/v1/users/me/messages".

- [Método](https://developers.google.com/workspace/gmail/api/reference/rest/v1/users.messages/get?hl=pt-br): Ação que você quer executar, por exemplo, GET (listar), POST (criar) ou DELETE (remover).

- Parâmetros: São as informações adicionais da requisição, por exemplo, maxResults=32.

- Autenticação: Prova quem você é, por exemplo, Token OAuth 2.0.

- Resposta: Dados retornados pela API, geralmente em formato JSON.

Nesse texto será mostrado todas essas componentes e mostrado como se trabalha com a API do Gmail.


## Preparação do Ambiente

Para iniciar o fluxo de autenticação e de autorização é necessário:

- Python 3.10.7 ou mais recente
- A ferramenta de gerenciamento de pacotes pip
- Uma Conta do Google com o Gmail ativado.
- Um projeto do Google Cloud.


### Como criar um projeto no google cloud

Esse projeto organizará as APIs que você usa, as credenciais de acesso e o faturamento.


1. Com seu gmail logado, acesse [Google Cloud](https://console.cloud.google.com/)

2. No topo da página, ao lado direito do logotipo do Google Cloud, clique no seletor de projetos (que pode estar escrito "Selecione um projeto" ou mostrando o nome de um projeto atual) . No menu que abrir, clique em "Novo Projeto".

3. Configure os Detalhes do Projeto:

- Uma tela chamada "Novo Projeto" será exibida. Preencha os campos :

  - Nome do projeto: Escolha um nome descritivo (ex: "Meu Projeto Gmail API"). É com esse nome que você o identificará no console.

  - Local (Organização): Se sua conta estiver associada a uma organização, você poderá escolher uma pasta. Para uso pessoal, a opção "Sem organização" é a mais comum.

O ID do Projeto será gerado automaticamente a partir do nome, mas você pode clicar em "Editar" para personalizá-lo. Atenção: este ID é único e não poderá ser alterado depois .

4. Clique no botão "Criar". 

5. Selecione o Novo Projeto: Após a criação, uma notificação aparecerá. Clique em "Selecionar projeto" para ser redirecionado para o painel do seu novo projeto . Verifique se o nome do seu projeto está visível no seletor de projetos no topo da página (Ao lado direito do logotipo do google cloud).

### Ativação da API

É necessário ativar a API para que o script Python consiga se conectar com o Gmail.



Dentro do seu [projeto](https://console.cloud.google.com/apis/enableflow?apiid=gmail.googleapis.com&hl=pt-br), vá em ** "APIs e Serviços" > "Biblioteca"**

Procure por "Gmail API" e clique em "Ativar"

### Consentimento OAuth

É necessário ativar a tela de consentimento OAuth para que a API do Gmail de identifique quando necessário.

Na barra lateral esquerda do Google Cloud Platform clique em  *"APIs e Serviços" > "Tela de consentimento OAuth"*.

Escolha "Interno" e preencha as informações básicas como o nome do app, e-mail de suporte,

### Criação das Credenciais

As credenciais identificam seu script para o Google, permitindo que ele solicite autorização do usuário para acessar os dados do Gmail. Sem ele, o fluxo de autenticação OAuth 2.0 não pode ser iniciado, pois o Google não sabe qual aplicativo está pedindo acesso. 

Com credentials.json seu script consegue pedir autorização, gerar o token.json e finalmente acessar a API. É como um documento de identidade do seu aplicativo.

Na mesma tela, "APIs e Serviços", vá em:

**"Credenciais" e clique em "+ Criar Credenciais" > "ID do cliente OAuth"**

Escolha "Aplicativo da Web" e adicione um nome.

Após criar, baixe o arquivo JSON. 

Renomeie esse arquivo para credentials.json e coloque-o na raiz do seu projeto. 

É este arquivo que seu código lê quando chama 

```bash 
InstalledAppFlow.from_client_secrets_file()
```

## Criação do script Python

Inicie instalando a biblioteca de cliente do Google para o Python.

Digite em seu terminal:

```bash
python3 -m pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
```

In [26]:
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# O escopo concede ao seu aplicativo acesso somente leitura a todos os recursos da caixa de correio do usuário. 
# Com ele, seu script pode listar e-mails e seus IDs, baixar mensagens e seus metadados, 
# mas não pode enviar ou excluir mensagens.
# Obs: Se modificar os escopos, exclua o arquivo token.json.
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

"""Mostra o uso básico da API do Gmail."""
  

creds = None

# O arquivo token.json armazena as credenciais de acesso do usuário. 
# Ele é criado automaticamente quando o fluxo de autorização é concluído pela primeira vez.
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

# Se as credenciais forem inválidas ou expiradas, solicita login do usuário.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json", SCOPES
        )
        creds = flow.run_local_server(port=0)

# Salva as credenciais para a próxima execução
    with open("token.json", "w") as token:
        token.write(creds.to_json())

try:
# Chama a API do Gmail
    service = build("gmail", "v1", credentials=creds)

except HttpError as error:
    print(f"An error occurred: {error}")

In [29]:
 # Lista os rótulos da caixa de correio do usuário.    
results_labels = service.users().labels().list(userId="me").execute()

print(f'Resultados: {results_labels}')

# Obtém a lista de rótulos. Se não houver rótulos, a lista será vazia.
labels = results_labels.get("labels", [])
if labels:
    print("Labels:")

    for label in labels:
        print(label["name"])
else:
    print("No labels found.")    

Resultados: {'labels': [{'id': 'CHAT', 'name': 'CHAT', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelShow', 'type': 'system'}, {'id': 'SENT', 'name': 'SENT', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelShow', 'type': 'system'}, {'id': 'INBOX', 'name': 'INBOX', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelShow', 'type': 'system'}, {'id': 'IMPORTANT', 'name': 'IMPORTANT', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelShow', 'type': 'system'}, {'id': 'TRASH', 'name': 'TRASH', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelHide', 'type': 'system'}, {'id': 'DRAFT', 'name': 'DRAFT', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelShow', 'type': 'system'}, {'id': 'SPAM', 'name': 'SPAM', 'messageListVisibility': 'hide', 'labelListVisibility': 'labelHide', 'type': 'system'}, {'id': 'CATEGORY_FORUMS', 'name': 'CATEGORY_FORUMS', 'type': 'system'}, {'id': 'CATEGORY_UPDATES', 'name': 'CATEGORY_UPDATES', 

In [31]:
results_messages = service.users().messages().list(userId="me").execute()
mensagens = results_messages.get("messages", [])

print('Resultados')
if results_messages:
    for message in mensagens:
        print(message["id"])
else:
    print("No messages found.")

Resultados
19c77030507d67e4
19c76f58fb69c313
19c76f473750d0cb
19c76f17b6d4e83e
19c76dd03ae935e0
19c76c630cb807fc
19c76c5e92eba809
19c76728093a8c0c
19c7668018c00067
19c7606143806e31
19c76016f428a9be
19c75f9811c1a9b7
19c75e3366dc024a
19c75e312497660a
19c75bd153953627
19c75a7f89f75bbc
19c7575790cdac8a
19c75744cfff92ab
19c74abcc06fa964
19c7426fcce79689
19c734bae7b66a7a
19c73459dce86193
19c73148a0e715b4
19c730c7ae35fe46
19c72f41dc52827b
19c72db7c7f72bbb
19c72db67f61f702
19c72db62fdd3952
19c72b7c70b05122
19c728f5ff755167
19c72210be29fd8e
19c722010aba508c
19c7207dc792ad80
19c7205ec2e7e2d2
19c71e242f71cf85
19c71cbc86671300
19c71bf71f8dd4cd
19c71b71180b3336
19c71b6c158a38e1
19c71b10e948e3f2
19c71ac5f70a575e
19c71931ad6c86b7
19c70f56c5e738de
19c70dcfb9289a50
19c709ff04f55864
19c7081abf6bc473
19c70740dde7130a
19c6ffe368c9e53d
19c6fcc9c4b7ada9
19c6f4f7d51b4e68
19c6e45cf42ddf98
19c6e39538a0f6f6
19c6e252d95d9e48
19c6ddb5b9e6ccd7
19c6dc4463c0be1b
19c6db6afc2c0542
19c6d7bb6a3c63e1
19c6d47a498ff705
19c

#